# Berrar P1 — Hybrid EPL Colab Runner (Direct Folder Version)

Use this notebook when the implementation has already been uploaded to Google Drive as an **extracted folder**, not as a ZIP.

Expected default location:

```text
My Drive/ml_project/code/data_pipeline/extracted_p1_final/md1_data_pipeline_code/
```

The notebook validates the P1 files before import and imports P1 symbols from their concrete modules.


## 1. Configuration

In [1]:
# Path relative to My Drive.
CODE_FOLDER = "ml_project/code/data_pipeline/extracted/md1_data_pipeline_code"

RUN_SYNTHETIC_PREFLIGHT = True
P1_RUN_MODE = "real"                 # "real" or "synthetic_demo"
P1_FULL_RECENCY_SEARCH = True        # real method: n = 9..100
P1_RUN_KNN_SEARCH = False            # optional / expensive
P1_CUSTOM_RUN_NAME = None
OVERWRITE_DOWNLOADS = False

# Optional existing MD1 gold_prematch.parquet for Experiment C.
BASELINE_GOLD_PREMATCH_PATH = None


## 2. Mount Drive and validate the uploaded folder

In [2]:
import os
import sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_MYDRIVE = Path("/content/drive/MyDrive")
PACKAGE_ROOT = DRIVE_MYDRIVE / CODE_FOLDER
PACKAGE_DIR = PACKAGE_ROOT / "md1_data_pipeline"

if not PACKAGE_ROOT.exists():
    raise FileNotFoundError(
        "Code folder not found:\n"
        f"{PACKAGE_ROOT}\n\n"
        "Update CODE_FOLDER in the configuration cell to the folder you uploaded."
    )

required_files = [
    PACKAGE_DIR / "__init__.py",
    PACKAGE_DIR / "core.py",
    PACKAGE_DIR / "workflow.py",
    PACKAGE_DIR / "p1_berrar.py",
    PACKAGE_DIR / "p1_hybrid.py",
    PACKAGE_ROOT / "requirements_colab.txt",
]

missing = [str(p) for p in required_files if not p.exists()]

if missing:
    raise RuntimeError(
        "The uploaded folder is not the complete P1 hybrid implementation.\n\n"
        "Missing files:\n  - "
        + "\n  - ".join(missing)
        + "\n\n"
        "Make sure you uploaded the patched/final folder, not the old base code folder."
    )

print("Code folder:", PACKAGE_ROOT)
print("Package folder:", PACKAGE_DIR)
print("P1 implementation files detected: PASS")

for p in required_files:
    print("  OK:", p.relative_to(PACKAGE_ROOT))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code folder: /content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipeline_code
Package folder: /content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipeline_code/md1_data_pipeline
P1 implementation files detected: PASS
  OK: md1_data_pipeline/__init__.py
  OK: md1_data_pipeline/core.py
  OK: md1_data_pipeline/workflow.py
  OK: md1_data_pipeline/p1_berrar.py
  OK: md1_data_pipeline/p1_hybrid.py
  OK: requirements_colab.txt


## 3. Install dependencies, compile, and import the P1 modules

In [3]:
import subprocess
import py_compile
import importlib

requirements_path = PACKAGE_ROOT / "requirements_colab.txt"

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade-strategy",
    "only-if-needed",
    "-r",
    str(requirements_path),
])

# Compile package source first.
for module_path in sorted(PACKAGE_DIR.glob("*.py")):
    py_compile.compile(str(module_path), doraise=True)

# Put this exact uploaded folder first on sys.path.
PACKAGE_ROOT_STR = str(PACKAGE_ROOT)
sys.path = [p for p in sys.path if p != PACKAGE_ROOT_STR]
sys.path.insert(0, PACKAGE_ROOT_STR)

# Remove any earlier/stale md1_data_pipeline imports from this Colab runtime.
for module_name in list(sys.modules):
    if module_name == "md1_data_pipeline" or module_name.startswith("md1_data_pipeline."):
        del sys.modules[module_name]

# Import P1 symbols from the modules that define them.
from md1_data_pipeline.p1_berrar import (
    PAPER_TOTAL_FEATURES,
    PAPER_HOMEAWAY_FEATURES,
    merge_existing_prematch_with_p1,
)
from md1_data_pipeline.workflow import (
    build_p1_default_config,
    execute_p1_workflow,
)

import md1_data_pipeline

print("Loaded package from:")
print(md1_data_pipeline.__file__)
print()
print("P1 total feature count:", len(PAPER_TOTAL_FEATURES))
print("P1 homeaway feature count:", len(PAPER_HOMEAWAY_FEATURES))

assert len(PAPER_TOTAL_FEATURES) == 6
assert len(PAPER_HOMEAWAY_FEATURES) == 18

print("Compilation and P1 import: PASS")


Loaded package from:
/content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipeline_code/md1_data_pipeline/__init__.py

P1 total feature count: 6
P1 homeaway feature count: 18
Compilation and P1 import: PASS


## 4. Synthetic P1 preflight

In [4]:
import pandas as pd
from IPython.display import display

RESULTS_ROOT = PACKAGE_ROOT / "colab_results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

if RUN_SYNTHETIC_PREFLIGHT:
    _, smoke_cfg = build_p1_default_config(
        results_root=RESULTS_ROOT,
        run_mode="synthetic_demo",
        run_name="P1_Hybrid_Synthetic_Preflight",
    )

    smoke = execute_p1_workflow(config=smoke_cfg)

    print("Synthetic summary")
    display(pd.Series(smoke["summary"], name="value").to_frame())

    print("Leakage/source tests")
    display(smoke["leakage_tests"][["test", "passed", "details"]])

    print("Readiness")
    display(smoke["readiness"][["check", "passed", "details"]])

    assert smoke["leakage_tests"]["passed"].all()
    assert smoke["readiness"]["passed"].all()

    assert len([c for c in PAPER_TOTAL_FEATURES if c in smoke["selected_total"].columns]) == 6
    assert len([c for c in PAPER_HOMEAWAY_FEATURES if c in smoke["selected_homeaway"].columns]) == 18

    print("Synthetic P1 preflight: PASS")
else:
    print("Synthetic preflight skipped.")


Synthetic summary


,value
paper,"Berrar, Lopes & Dubitzky (2024)"
design,TA-approved Football-Data history + StatsBomb ...
data_mode,synthetic_demo
history_provider,football_data
history_seasons,"[2012/2013, 2013/2014, 2014/2015]"
history_season_count,3
history_match_count,90
target_provider,statsbomb
target_season,2015/2016
target_match_count,30


Leakage/source tests


,test,passed,details
0,pre-match source time contract,True,all source finishes precede target kickoff
1,target-result perturbation,True,target 800015 unchanged
2,future-result perturbation,True,future 800029 did not alter earlier 800010
3,historical positive control,True,allowed old history match -15010 changes later...
4,source-role separation,True,pre-target Football-Data history and StatsBomb...
5,forbidden historical fields,True,canonical P1 history carries no odds/shots/car...
6,season-boundary continuity,True,team 101 carries history 2012/2013->2013/2014
7,gap continuity,True,team 106 retains same-league history across ga...
8,minimum-six rule,True,>=6 prior total matches per team
9,selected-n isolation,True,selected n=9 unchanged after non-train label m...


Readiness


,check,passed,details
0,TA-approved source design,True,TA approved Football-Data historical Premier L...
1,historical season completeness,True,3/3 historical seasons complete
2,StatsBomb target completeness,True,target rows=30
3,target examples are StatsBomb only,True,all supervised P1 target rows are StatsBomb
4,Football-Data target results excluded,True,history ends 2014/2015
5,cross-source target identities,True,12 mapping rows refer to target clubs
6,chronological target split,True,StatsBomb target matches sorted chronologically
7,6-feature total contract,True,exactly six
8,18-feature homeaway contract,True,exactly eighteen
9,normalized rank bounded,True,"all observed ranks in [0,1]"


Synthetic P1 preflight: PASS


## 5. Configure and run the real P1 pipeline

In [5]:
P1_RUN_NAME, p1_config = build_p1_default_config(
    results_root=RESULTS_ROOT,
    run_mode=P1_RUN_MODE,
    run_name=P1_CUSTOM_RUN_NAME,
    full_recency_search=P1_FULL_RECENCY_SEARCH,
)

p1_config.overwrite_downloads = OVERWRITE_DOWNLOADS
p1_config.p1_run_knn_search = P1_RUN_KNN_SEARCH

print("Run name:", P1_RUN_NAME)
print("Output root:", p1_config.project_root)
print("History provider:", p1_config.p1_history_provider)
print(
    "Historical seasons:",
    p1_config.p1_history_start_season,
    "through",
    p1_config.p1_history_end_season,
)
print(
    "StatsBomb target:",
    p1_config.competition_name,
    p1_config.season_name,
)
print(
    "Recency search:",
    p1_config.p1_recency_min,
    "..",
    p1_config.p1_recency_max,
)

p1 = execute_p1_workflow(config=p1_config)

print()
print("Runtime (min):", round(p1["summary"]["runtime_seconds"] / 60, 2))
print("Selected n:", p1["selected_n"])
print("Readiness passed:", p1["summary"]["readiness_passed"])

if not p1["summary"]["readiness_passed"]:
    raise AssertionError("P1 readiness failed. Do not use the generated results.")


Run name: EPL_2000_01_2015_16_Berrar_P1_HYBRID_REAL
Output root: /content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipeline_code/colab_results/EPL_2000_01_2015_16_Berrar_P1_HYBRID_REAL
History provider: football_data
Historical seasons: 2000/2001 through 2014/2015
StatsBomb target: Premier League 2015/2016
Recency search: 9 .. 100

Runtime (min): 0.57
Selected n: 28
Readiness passed: True


## 6. Inspect real-run evidence

In [6]:
print("Run summary")
display(pd.Series(p1["summary"], name="value").to_frame())

print("Historical season completeness")
display(p1["history_season_audit"])

print("StatsBomb target-season completeness")
display(p1["target_season_audit"])

print("Pearson recency search — top candidates")
display(
    p1["pearson_audit"]
    .sort_values("pearson_correlation", ascending=False)
    .head(20)
)

print("History coverage")
display(p1["history_coverage"])

print("Common comparison population")
display(p1["common_population"])

print("Leakage/source-contamination tests")
display(p1["leakage_tests"])

print("Readiness")
display(p1["readiness"])

print("Paper fidelity / project adaptations")
display(p1["fidelity"])

assert p1["leakage_tests"]["passed"].all()
assert p1["readiness"]["passed"].all()

print("ALL P1 TESTS / READINESS CHECKS: PASS")


Run summary


,value
paper,"Berrar, Lopes & Dubitzky (2024)"
design,TA-approved Football-Data history + StatsBomb ...
data_mode,real
history_provider,football_data
history_seasons,"(2000/2001, 2001/2002, 2002/2003, 2003/2004, 2..."
history_season_count,15
history_match_count,5700
target_provider,statsbomb
target_season,2015/2016
target_match_count,380


Historical season completeness


,season_name,provider,rows,complete_rows,team_count,expected_team_count,expected_matches,duplicate_directed_fixtures,min_matches_per_team,max_matches_per_team,unordered_pair_count,pairs_with_wrong_count,missing_core_values,passed
0,2000/2001,football_data_history,380,380,20,20,380,0,38,38,190,0,0,True
1,2001/2002,football_data_history,380,380,20,20,380,0,38,38,190,0,0,True
2,2002/2003,football_data_history,380,380,20,20,380,0,38,38,190,0,0,True
3,2003/2004,football_data_history,380,380,20,20,380,0,38,38,190,0,0,True
4,2004/2005,football_data_history,380,380,20,20,380,0,38,38,190,0,0,True
5,2005/2006,football_data_history,380,380,20,20,380,0,38,38,190,0,0,True
6,2006/2007,football_data_history,380,380,20,20,380,0,38,38,190,0,0,True
7,2007/2008,football_data_history,380,380,20,20,380,0,38,38,190,0,0,True
8,2008/2009,football_data_history,380,380,20,20,380,0,38,38,190,0,0,True
9,2009/2010,football_data_history,380,380,20,20,380,0,38,38,190,0,0,True


StatsBomb target-season completeness


...

Readiness


,check,passed,details
0,TA-approved source design,True,TA approved Football-Data historical Premier L...
1,historical season completeness,True,15/15 historical seasons complete
2,StatsBomb target completeness,True,target rows=380
3,target examples are StatsBomb only,True,all supervised P1 target rows are StatsBomb
4,Football-Data target results excluded,True,history ends 2014/2015
5,cross-source target identities,True,39 mapping rows refer to target clubs
6,chronological target split,True,StatsBomb target matches sorted chronologically
7,6-feature total contract,True,exactly six
8,18-feature homeaway contract,True,exactly eighteen
9,normalized rank bounded,True,"all observed ranks in [0,1]"


Paper fidelity / project adaptations


,paper_component,paper_method,our_implementation,status,reason_for_adaptation
0,feature source,basic complete match records,Football-Data complete EPL results before 2015...,project_required_adaptation,TA explicitly approved Football-Data historica...
1,super league,all seasons of one league concatenated chronol...,EPL 2000/01--2014/15 history + causal EPL 2015...,exact,same-league continuity preserved
2,modeling examples,paper uses generated league datasets,final supervised examples remain StatsBomb EPL...,project_required_adaptation,keeps StatsBomb as project primary modeling so...
3,season boundaries,team histories continue,same,exact,
4,team-history gaps,same-league gaps ignored,same; no lower-division matches inserted,exact,
5,minimum prior history,6 prior matches per team,6,exact,
6,aggregation,mean,mean,exact,
7,performance categories,"goals scored, conceded, league success",same,exact,
8,total feature set,6,6,exact,
9,homeaway feature set,18,18,exact,


ALL P1 TESTS / READINESS CHECKS: PASS


## 7. Optional merge with the existing MD1 pre-match table

In [7]:
if BASELINE_GOLD_PREMATCH_PATH:
    baseline_path = Path(BASELINE_GOLD_PREMATCH_PATH)

    if not baseline_path.exists():
        raise FileNotFoundError(f"Baseline Gold table not found: {baseline_path}")

    baseline = pd.read_parquet(baseline_path)

    augmented = merge_existing_prematch_with_p1(
        baseline,
        p1["selected_homeaway"],
        common_only=True,
    )

    output_path = p1_config.gold_root / "md1_plus_p1_homeaway_common.parquet"
    augmented.to_parquet(output_path, index=False)

    print("Augmented Gold table:", output_path)
    print("Rows:", len(augmented))
else:
    print("BASELINE_GOLD_PREMATCH_PATH is not set; merge skipped.")


BASELINE_GOLD_PREMATCH_PATH is not set; merge skipped.


## 8. Generated artifacts

In [8]:
print("P1 result root:", p1_config.project_root)
print()
print("Important outputs:")
print("  ", p1_config.gold_root / "p1_features_total_selected.parquet")
print("  ", p1_config.gold_root / "p1_features_homeaway_selected.parquet")
print("  ", p1_config.audit_root / "p1_pearson_recency_search.csv")
print("  ", p1_config.audit_root / "p1_season_completeness.csv")
print("  ", p1_config.audit_root / "p1_target_season_completeness.csv")
print("  ", p1_config.audit_root / "p1_team_identity_audit.csv")
print("  ", p1_config.audit_root / "p1_history_coverage.csv")
print("  ", p1_config.audit_root / "p1_leakage_tests.csv")
print("  ", p1_config.audit_root / "p1_readiness.csv")
print("  ", p1_config.audit_root / "p1_fidelity_table.csv")
print("  ", p1_config.audit_root / "p1_run_summary.json")


P1 result root: /content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipeline_code/colab_results/EPL_2000_01_2015_16_Berrar_P1_HYBRID_REAL

Important outputs:
   /content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipeline_code/colab_results/EPL_2000_01_2015_16_Berrar_P1_HYBRID_REAL/data/gold/p1_features_total_selected.parquet
   /content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipeline_code/colab_results/EPL_2000_01_2015_16_Berrar_P1_HYBRID_REAL/data/gold/p1_features_homeaway_selected.parquet
   /content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipeline_code/colab_results/EPL_2000_01_2015_16_Berrar_P1_HYBRID_REAL/audit/p1_pearson_recency_search.csv
   /content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipeline_code/colab_results/EPL_2000_01_2015_16_Berrar_P1_HYBRID_REAL/audit/p1_season_completeness.csv
   /content/drive/MyDrive/ml_project/code/data_pipeline/extracted/md1_data_pipel

## If the import check fails

If Cell 2 reports that `p1_berrar.py` or `p1_hybrid.py` is missing, the folder uploaded to Drive is the **old base code**.

In that case, replace:

```text
My Drive/ml_project/code/data_pipeline/extracted_p1_final/md1_data_pipeline_code/
```

with the final patched implementation folder, then restart the Colab runtime and run this notebook from the top.
